# DQ Engine Fix & Silver Layer Rule Additions
Fixes DQ_STATUS not updating in Bronze, adds GN_InList rule, adds Silver DQ rules, and rebuilds the DQ engine procedure.

*Co-authored with CoCo*

In [ ]:
%%sql -r use_db
USE DATABASE P360_DQ;

In [ ]:
%%sql -r add_inlist_rule
-- Add GN_InList rule (validates column value is in a specified list)
INSERT INTO CONFIG.DQ_RULE (CATEGORY_ID, RULE_CODE, RULE_DESC, RULE_EXP, RULE_CATEGORY)
SELECT 6, 'GN_InList', 'Validate value is in allowed list',
       'CASE WHEN ${INPUT1} NOT IN (${INPUT2}) THEN ''FAIL'' ELSE ''PASS'' END',
       'SIMPLE'
WHERE NOT EXISTS (SELECT 1 FROM CONFIG.DQ_RULE WHERE RULE_CODE = 'GN_InList');

In [ ]:
%%sql -r inlist_rule_id
-- Capture the new RULE_ID for GN_InList
SET v_inlist_rule_id = (SELECT RULE_ID FROM CONFIG.DQ_RULE WHERE RULE_CODE = 'GN_InList');
SELECT $v_inlist_rule_id AS INLIST_RULE_ID;

In [ ]:
%%sql -r bronze_credential_feed
-- Bronze: Credential must be in valid list
INSERT INTO CONFIG.DQ_FEED (LAYER, DOMAIN, RULE_ID, TABLE_NM, RECORD_KEY_NM,
    INCREMENTAL_DATE_COLUMN, DQ_RULE_INPUT, DQ_RULE_INPUT_WHERE_COL,
    CRITICALITY_IND, ACTIVE_IND, EXECUTION_GROUP)
SELECT 'BRONZE', 'PROVIDERS', (SELECT RULE_ID FROM CONFIG.DQ_RULE WHERE RULE_CODE = 'GN_InList'), 'STG_NPI_REGISTRY', 'NPI_SK',
       'LOAD_DT', 'CREDENTIAL', '''MD'',''DO'',''NP'',''PA'',''RN'',''APRN'',''DPM'',''DDS'',''PhD'',''PharmD'',''OD'',''DC''',
       'Y', 'Y', 'GROUP_A'
WHERE NOT EXISTS (
    SELECT 1 FROM CONFIG.DQ_FEED
    WHERE TABLE_NM = 'STG_NPI_REGISTRY' AND DQ_RULE_INPUT = 'CREDENTIAL'
      AND LAYER = 'BRONZE'
);

In [ ]:
%%sql -r silver_credential_feed
-- Silver: Credential must be in valid list
INSERT INTO CONFIG.DQ_FEED (LAYER, DOMAIN, RULE_ID, TABLE_NM, RECORD_KEY_NM,
    INCREMENTAL_DATE_COLUMN, DQ_RULE_INPUT, DQ_RULE_INPUT_WHERE_COL,
    CRITICALITY_IND, ACTIVE_IND, EXECUTION_GROUP)
SELECT 'SILVER', 'PROVIDERS', (SELECT RULE_ID FROM CONFIG.DQ_RULE WHERE RULE_CODE = 'GN_InList'), 'DIM_PROVIDER', 'PROVIDER_SK',
       'LOAD_DT', 'CREDENTIAL', '''MD'',''DO'',''NP'',''PA'',''RN'',''APRN'',''DPM'',''DDS'',''PhD'',''PharmD'',''OD'',''DC''',
       'Y', 'Y', 'GROUP_A'
WHERE NOT EXISTS (
    SELECT 1 FROM CONFIG.DQ_FEED
    WHERE TABLE_NM = 'DIM_PROVIDER' AND DQ_RULE_INPUT = 'CREDENTIAL'
      AND LAYER = 'SILVER'
);

In [ ]:
%%sql -r silver_gender_feed
-- Silver: Gender must be M or F
INSERT INTO CONFIG.DQ_FEED (LAYER, DOMAIN, RULE_ID, TABLE_NM, RECORD_KEY_NM,
    INCREMENTAL_DATE_COLUMN, DQ_RULE_INPUT, DQ_RULE_INPUT_WHERE_COL,
    CRITICALITY_IND, ACTIVE_IND, EXECUTION_GROUP)
SELECT 'SILVER', 'PROVIDERS', (SELECT RULE_ID FROM CONFIG.DQ_RULE WHERE RULE_CODE = 'GN_InList'), 'DIM_PROVIDER', 'PROVIDER_SK',
       'LOAD_DT', 'GENDER', '''M'',''F''',
       'N', 'Y', 'GROUP_A'
WHERE NOT EXISTS (
    SELECT 1 FROM CONFIG.DQ_FEED
    WHERE TABLE_NM = 'DIM_PROVIDER' AND DQ_RULE_INPUT = 'GENDER'
      AND LAYER = 'SILVER'
);

In [ ]:
%%sql -r silver_dedup_feed
-- Silver: Provider NPI must not be duplicate
INSERT INTO CONFIG.DQ_FEED (LAYER, DOMAIN, RULE_ID, TABLE_NM, RECORD_KEY_NM,
    INCREMENTAL_DATE_COLUMN, DQ_RULE_INPUT, CRITICALITY_IND, ACTIVE_IND, EXECUTION_GROUP)
SELECT 'SILVER', 'PROVIDERS', 1, 'DIM_PROVIDER', 'PROVIDER_SK',
       'LOAD_DT', 'PROVIDER_NPI', 'Y', 'Y', 'GROUP_A'
WHERE NOT EXISTS (
    SELECT 1 FROM CONFIG.DQ_FEED
    WHERE TABLE_NM = 'DIM_PROVIDER' AND RULE_ID = 1
      AND LAYER = 'SILVER' AND DQ_RULE_INPUT = 'PROVIDER_NPI'
);

In [ ]:
%%sql -r silver_phone_feed
-- Silver: Provider phone not null (non-critical)
INSERT INTO CONFIG.DQ_FEED (LAYER, DOMAIN, RULE_ID, TABLE_NM, RECORD_KEY_NM,
    INCREMENTAL_DATE_COLUMN, DQ_RULE_INPUT, CRITICALITY_IND, ACTIVE_IND, EXECUTION_GROUP)
SELECT 'SILVER', 'PROVIDERS', 2, 'DIM_PROVIDER', 'PROVIDER_SK',
       'LOAD_DT', 'PHONE', 'N', 'Y', 'GROUP_A'
WHERE NOT EXISTS (
    SELECT 1 FROM CONFIG.DQ_FEED
    WHERE TABLE_NM = 'DIM_PROVIDER' AND RULE_ID = 2 AND DQ_RULE_INPUT = 'PHONE'
      AND LAYER = 'SILVER'
);

In [ ]:
USE DATABASE P360_DQ;
USE SCHEMA CONFIG;

CREATE OR REPLACE PROCEDURE CONFIG.SP_RUN_DQ_CHECK(
    P_RUN_ID        VARCHAR,
    P_TABLE_NM      VARCHAR,
    P_LAYER         VARCHAR,
    P_DQ_BATCH_ID   VARCHAR
)
RETURNS VARIANT
LANGUAGE SQL
EXECUTE AS CALLER
AS
$$
DECLARE
    v_schema            STRING;
    v_fq_table          STRING;
    v_overall_count     NUMBER DEFAULT 0;
    v_fail_count        NUMBER DEFAULT 0;
    v_dq_start_ts       TIMESTAMP_NTZ;
    v_dq_end_ts         TIMESTAMP_NTZ;
    v_dyn_sql           STRING;
    v_resolved_exp      STRING;
    v_err_msg           STRING;
    v_record_key_nm     STRING;
    v_threshold_exceeded BOOLEAN DEFAULT FALSE;
    -- Feed cursor variables
    v_feed_id           NUMBER;
    v_rule_id           NUMBER;
    v_dq_rule_input     STRING;
    v_feed_rec_key      STRING;
    v_incr_col          STRING;
    v_criticality       STRING;
    v_domain            STRING;
    v_join_tbl          STRING;
    v_join_col          STRING;
    v_where_col         STRING;
    v_rule_exp          STRING;
    v_rule_category     STRING;
    v_rule_code         STRING;
BEGIN
    v_dq_start_ts := CURRENT_TIMESTAMP();

    v_schema := CASE
        WHEN :P_LAYER = 'BRONZE' THEN 'BRONZE'
        WHEN :P_LAYER = 'SILVER' THEN 'SILVER'
        WHEN :P_LAYER = 'GOLD'   THEN 'GOLD'
        ELSE :P_LAYER
    END;
    v_fq_table := 'P360_DQ.' || :v_schema || '.' || :P_TABLE_NM;

    -- Step 1: Count source rows needing evaluation
    v_dyn_sql := 'SELECT COUNT(*) AS CNT FROM ' || :v_fq_table || ' WHERE DQ_STATUS IS NULL';
    LET rs_cnt RESULTSET := (EXECUTE IMMEDIATE :v_dyn_sql);
    LET cur_cnt CURSOR FOR rs_cnt;
    FOR r IN cur_cnt DO
        v_overall_count := r.CNT;
    END FOR;

    IF (:v_overall_count = 0) THEN
        INSERT INTO P360_DQ.AUDIT.DQ_LOG (DQ_BATCH_ID, LAYER, DOMAIN, TABLE_NM, FEED_ID, DQ_START_TS, DQ_END_TS, OVERALL_COUNT, FAILED_COUNT, STATUS, UPDATED_TS)
        VALUES (:P_DQ_BATCH_ID, :P_LAYER, NULL, :P_TABLE_NM, NULL, :v_dq_start_ts, CURRENT_TIMESTAMP(), 0, 0, 'Success', CURRENT_TIMESTAMP());
        RETURN OBJECT_CONSTRUCT('status', 'COMPLETED', 'table', :P_TABLE_NM, 'total', 0, 'failed', 0, 'threshold_exceeded', FALSE);
    END IF;

    -- Get RECORD_KEY_NM
    SELECT RECORD_KEY_NM INTO v_record_key_nm
    FROM P360_DQ.CONFIG.DQ_FEED
    WHERE TABLE_NM = :P_TABLE_NM AND LAYER = :P_LAYER AND ACTIVE_IND = 'Y'
    LIMIT 1;

    -- Step 2: Process each feed via token-substituted RULE_EXP
    LET v_feeds RESULTSET := (
        SELECT f.FEED_ID AS F_FEED_ID, f.RULE_ID AS F_RULE_ID,
               f.DQ_RULE_INPUT AS F_DQ_RULE_INPUT, f.RECORD_KEY_NM AS F_RECORD_KEY_NM,
               f.INCREMENTAL_DATE_COLUMN AS F_INCR_COL, f.CRITICALITY_IND AS F_CRIT,
               f.DOMAIN AS F_DOMAIN,
               f.DQ_RULE_INPUT_JOIN_TBL AS F_JOIN_TBL,
               f.DQ_RULE_INPUT_JOIN_COL AS F_JOIN_COL,
               f.DQ_RULE_INPUT_WHERE_COL AS F_WHERE_COL,
               r.RULE_EXP AS R_RULE_EXP, r.RULE_CATEGORY AS R_RULE_CATEGORY,
               r.RULE_CODE AS R_RULE_CODE
        FROM P360_DQ.CONFIG.DQ_FEED f
        JOIN P360_DQ.CONFIG.DQ_RULE r ON f.RULE_ID = r.RULE_ID
        WHERE f.TABLE_NM = :P_TABLE_NM
          AND f.LAYER = :P_LAYER
          AND f.ACTIVE_IND = 'Y'
        ORDER BY f.FEED_ID
    );
    LET cur_feeds CURSOR FOR v_feeds;

    FOR feed IN cur_feeds DO
        BEGIN
            -- Assign cursor fields to local variables
            v_feed_id := feed.F_FEED_ID;
            v_rule_id := feed.F_RULE_ID;
            v_dq_rule_input := feed.F_DQ_RULE_INPUT;
            v_feed_rec_key := feed.F_RECORD_KEY_NM;
            v_incr_col := feed.F_INCR_COL;
            v_criticality := feed.F_CRIT;
            v_domain := feed.F_DOMAIN;
            v_join_tbl := feed.F_JOIN_TBL;
            v_join_col := feed.F_JOIN_COL;
            v_where_col := feed.F_WHERE_COL;
            v_rule_exp := feed.R_RULE_EXP;
            v_rule_category := feed.R_RULE_CATEGORY;
            v_rule_code := feed.R_RULE_CODE;

            -- Token substitution on RULE_EXP
            v_resolved_exp := :v_rule_exp;
            v_resolved_exp := REPLACE(:v_resolved_exp, '${INPUT1}', :v_dq_rule_input);
            v_resolved_exp := REPLACE(:v_resolved_exp, '${INPUT2}', COALESCE(:v_where_col, ''));
            v_resolved_exp := REPLACE(:v_resolved_exp, '${INPUTN}', :v_dq_rule_input);
            v_resolved_exp := REPLACE(:v_resolved_exp, '${TABLE_NAME}', :v_fq_table);
            v_resolved_exp := REPLACE(:v_resolved_exp, '${JOINTBL1}', :v_fq_table);
            v_resolved_exp := REPLACE(:v_resolved_exp, '${JOINTBL2}', COALESCE(:v_join_tbl, ''));
            v_resolved_exp := REPLACE(:v_resolved_exp, '${JOINCOL1}', :v_dq_rule_input);
            v_resolved_exp := REPLACE(:v_resolved_exp, '${JOINCOL2}', COALESCE(:v_join_col, ''));
            v_resolved_exp := REPLACE(:v_resolved_exp, '${WHERECOL1}', COALESCE(:v_where_col, ''));
            v_resolved_exp := REPLACE(:v_resolved_exp, '${INCREMENTAL_DATE_COLUMN}', 'DQ_STATUS IS NULL');

            IF (:v_rule_category = 'SIMPLE') THEN
                v_dyn_sql := 'INSERT INTO P360_DQ.AUDIT.DQ_RESULT ' ||
                    '(DQ_BATCH_ID, LAYER, DOMAIN, TABLE_NM, RULE_ID, FEED_ID, RECORD_KEY, RECORD_VALUE, RESULT, RECORD_INS_BY) ' ||
                    'SELECT ''' || :P_DQ_BATCH_ID || ''', ''' || :P_LAYER || ''', ''' || :v_domain || ''', ''' || :P_TABLE_NM || ''', ' ||
                    :v_rule_id || ', ' || :v_feed_id || ', ' ||
                    'CAST(' || :v_feed_rec_key || ' AS VARCHAR), ' ||
                    'CAST(' || :v_dq_rule_input || ' AS VARCHAR), ' ||
                    '''FAIL'', CURRENT_USER() ' ||
                    'FROM ' || :v_fq_table || ' ' ||
                    'WHERE DQ_STATUS IS NULL AND (' || :v_resolved_exp || ') = ''FAIL''';

            ELSEIF (:v_rule_category = 'COMPLEX') THEN
                v_dyn_sql := 'INSERT INTO P360_DQ.AUDIT.DQ_RESULT ' ||
                    '(DQ_BATCH_ID, LAYER, DOMAIN, TABLE_NM, RULE_ID, FEED_ID, RECORD_KEY, RECORD_VALUE, RESULT, RECORD_INS_BY) ' ||
                    'SELECT ''' || :P_DQ_BATCH_ID || ''', ''' || :P_LAYER || ''', ''' || :v_domain || ''', ''' || :P_TABLE_NM || ''', ' ||
                    :v_rule_id || ', ' || :v_feed_id || ', ' ||
                    'CAST(a.' || :v_feed_rec_key || ' AS VARCHAR), ' ||
                    'CAST(a.' || :v_dq_rule_input || ' AS VARCHAR), ' ||
                    :v_resolved_exp;

            ELSEIF (:v_rule_category = 'SQL_FEED') THEN
                v_dyn_sql := 'INSERT INTO P360_DQ.AUDIT.DQ_RESULT ' ||
                    '(DQ_BATCH_ID, LAYER, DOMAIN, TABLE_NM, RULE_ID, FEED_ID, RECORD_KEY, RECORD_VALUE, RESULT, RECORD_INS_BY) ' ||
                    'SELECT ''' || :P_DQ_BATCH_ID || ''', ''' || :P_LAYER || ''', ''' || :v_domain || ''', ''' || :P_TABLE_NM || ''', ' ||
                    :v_rule_id || ', ' || :v_feed_id || ', ' ||
                    'CAST(' || :v_feed_rec_key || ' AS VARCHAR), ' ||
                    'CAST(' || :v_dq_rule_input || ' AS VARCHAR), ' ||
                    '''FAIL'', CURRENT_USER() ' ||
                    'FROM (' || :v_resolved_exp || ') sub ' ||
                    'WHERE sub.RESULT = ''FAIL''';

            END IF;

            EXECUTE IMMEDIATE :v_dyn_sql;

        EXCEPTION
            WHEN OTHER THEN
                v_err_msg := SQLERRM;
                INSERT INTO P360_DQ.AUDIT.DQ_LOG
                    (DQ_BATCH_ID, LAYER, DOMAIN, TABLE_NM, FEED_ID, DQ_START_TS, DQ_END_TS, ERROR_MESSAGE, OVERALL_COUNT, FAILED_COUNT, STATUS, UPDATED_TS)
                VALUES (:P_DQ_BATCH_ID, :P_LAYER, :v_domain, :P_TABLE_NM, :v_feed_id,
                        :v_dq_start_ts, CURRENT_TIMESTAMP(), :v_err_msg, 0, 0, 'Fail', CURRENT_TIMESTAMP());
        END;
    END FOR;

    -- Step 3: Count total failures
    SELECT COUNT(*) INTO v_fail_count
    FROM P360_DQ.AUDIT.DQ_RESULT
    WHERE DQ_BATCH_ID = :P_DQ_BATCH_ID AND TABLE_NM = :P_TABLE_NM AND RESULT = 'FAIL';

    -- Step 4: MERGE DQ_STATUS back to source table
    v_dyn_sql := 'MERGE INTO ' || :v_fq_table || ' tgt USING (' ||
        'SELECT DISTINCT r.RECORD_KEY, ' ||
        'CASE WHEN SUM(CASE WHEN f.CRITICALITY_IND = ''Y'' THEN 1 ELSE 0 END) > 0 THEN ''FAIL'' ' ||
        'ELSE ''PASS-SOFT'' END AS COMPUTED_STATUS ' ||
        'FROM P360_DQ.AUDIT.DQ_RESULT r ' ||
        'JOIN P360_DQ.CONFIG.DQ_FEED f ON r.FEED_ID = f.FEED_ID ' ||
        'WHERE r.DQ_BATCH_ID = ''' || :P_DQ_BATCH_ID || ''' AND r.TABLE_NM = ''' || :P_TABLE_NM || ''' ' ||
        'GROUP BY r.RECORD_KEY' ||
        ') src ON CAST(tgt.' || :v_record_key_nm || ' AS VARCHAR) = src.RECORD_KEY ' ||
        'WHEN MATCHED AND tgt.DQ_STATUS IS NULL THEN UPDATE SET DQ_STATUS = src.COMPUTED_STATUS';
    EXECUTE IMMEDIATE :v_dyn_sql;

    -- Remaining NULL records = PASS
    v_dyn_sql := 'UPDATE ' || :v_fq_table || ' SET DQ_STATUS = ''PASS'' WHERE DQ_STATUS IS NULL';
    EXECUTE IMMEDIATE :v_dyn_sql;

    -- Step 5: Log results
    v_dq_end_ts := CURRENT_TIMESTAMP();

    INSERT INTO P360_DQ.AUDIT.DQ_LOG
        (DQ_BATCH_ID, LAYER, DOMAIN, TABLE_NM, FEED_ID, DQ_START_TS, DQ_END_TS, OVERALL_COUNT, FAILED_COUNT, STATUS, UPDATED_TS)
    VALUES (:P_DQ_BATCH_ID, :P_LAYER, NULL, :P_TABLE_NM, NULL, :v_dq_start_ts, :v_dq_end_ts, :v_overall_count, :v_fail_count, 'Success', CURRENT_TIMESTAMP());

    INSERT INTO P360_DQ.AUDIT.DQ_LOG_HISTORY
        (DQ_BATCH_ID, LAYER, DOMAIN, TABLE_NM, FEED_ID, DQ_START_TS, DQ_END_TS, OVERALL_COUNT, FAILED_COUNT, STATUS, UPDATED_TS)
    VALUES (:P_DQ_BATCH_ID, :P_LAYER, NULL, :P_TABLE_NM, NULL, :v_dq_start_ts, :v_dq_end_ts, :v_overall_count, :v_fail_count, 'Success', CURRENT_TIMESTAMP());

    -- Step 6: Check threshold
    IF (:v_overall_count > 0) THEN
        CALL P360_DQ.CONFIG.SP_CHECK_DQ_THRESHOLD(:P_RUN_ID, NULL, :P_TABLE_NM, :v_overall_count, :v_fail_count);
        v_threshold_exceeded := (SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID())));
    END IF;

    RETURN OBJECT_CONSTRUCT(
        'status', CASE WHEN :v_threshold_exceeded THEN 'THRESHOLD_EXCEEDED' ELSE 'COMPLETED' END,
        'table', :P_TABLE_NM,
        'layer', :P_LAYER,
        'total_records', :v_overall_count,
        'failed_records', :v_fail_count,
        'fail_pct', CASE WHEN :v_overall_count > 0 THEN ROUND((:v_fail_count::FLOAT / :v_overall_count) * 100, 2) ELSE 0 END,
        'threshold_exceeded', :v_threshold_exceeded
    );

EXCEPTION
    WHEN OTHER THEN
        v_err_msg := SQLERRM;
        INSERT INTO P360_DQ.AUDIT.DQ_LOG
            (DQ_BATCH_ID, LAYER, DOMAIN, TABLE_NM, FEED_ID, DQ_START_TS, DQ_END_TS, ERROR_MESSAGE, OVERALL_COUNT, FAILED_COUNT, STATUS, UPDATED_TS)
        VALUES (:P_DQ_BATCH_ID, :P_LAYER, NULL, :P_TABLE_NM, NULL, :v_dq_start_ts, CURRENT_TIMESTAMP(), :v_err_msg, 0, 0, 'Fail', CURRENT_TIMESTAMP());
        RETURN OBJECT_CONSTRUCT('status', 'FAILED', 'table', :P_TABLE_NM, 'error', :v_err_msg);
END;
$$;

In [ ]:
-- Reset DQ_STATUS so engine can re-evaluate all records
UPDATE P360_DQ.BRONZE.STG_NPI_REGISTRY SET DQ_STATUS = NULL;
UPDATE P360_DQ.BRONZE.STG_SPECIALTY_TYPE SET DQ_STATUS = NULL;
TRUNCATE TABLE P360_DQ.AUDIT.DQ_RESULT;
TRUNCATE TABLE P360_DQ.AUDIT.DQ_LOG;

In [ ]:
%%sql -r clear_audit
-- Clear previous DQ results for clean re-run
TRUNCATE TABLE P360_DQ.AUDIT.DQ_RESULT;
TRUNCATE TABLE P360_DQ.AUDIT.DQ_LOG;

In [ ]:
%%sql -r run_full_pipeline
-- Run full pipeline
CALL P360_DQ.ORCHESTRATION.SP_RUN_PACKAGE('FULL');

## Validation Queries
Run these after the pipeline completes to verify the fixes worked.

In [ ]:
%%sql -r bronze_npi_status
-- Check DQ_STATUS populated in Bronze
SELECT DQ_STATUS, COUNT(*) AS cnt FROM P360_DQ.BRONZE.STG_NPI_REGISTRY GROUP BY DQ_STATUS;

In [ ]:
%%sql -r bronze_spec_status
SELECT DQ_STATUS, COUNT(*) AS cnt FROM P360_DQ.BRONZE.STG_SPECIALTY_TYPE GROUP BY DQ_STATUS;

In [ ]:
%%sql -r dq_result_summary
-- Check DQ_RESULT has entries
SELECT TABLE_NM, RULE_ID, RESULT, COUNT(*) AS cnt
FROM P360_DQ.AUDIT.DQ_RESULT
GROUP BY TABLE_NM, RULE_ID, RESULT
ORDER BY TABLE_NM, RULE_ID;

In [ ]:
%%sql -r dim_provider_count
-- Check Silver DIM_PROVIDER is populated
SELECT COUNT(*) AS dim_provider_count FROM P360_DQ.SILVER.DIM_PROVIDER;

In [ ]:
%%sql -r dim_provider_sample
SELECT * FROM P360_DQ.SILVER.DIM_PROVIDER LIMIT 10;

In [ ]:
%%sql -r all_dq_feeds
-- Full DQ rules configuration
SELECT f.FEED_ID, f.LAYER, f.TABLE_NM, f.DQ_RULE_INPUT, f.DQ_RULE_INPUT_WHERE_COL, r.RULE_CODE, f.CRITICALITY_IND
FROM P360_DQ.CONFIG.DQ_FEED f
JOIN P360_DQ.CONFIG.DQ_RULE r ON f.RULE_ID = r.RULE_ID
WHERE f.ACTIVE_IND = 'Y'
ORDER BY f.LAYER, f.TABLE_NM, f.FEED_ID;

In [ ]:
%%sql -r debug_dq_log
-- Debug: Check DQ_LOG for any entries from the engine
SELECT * FROM P360_DQ.AUDIT.DQ_LOG ORDER BY DQ_START_TS DESC LIMIT 20;

In [ ]:
%%sql -r debug_step_log
-- Debug: Check step run log to see if DQ steps ran
SELECT STEP_NAME, STEP_STATUS, ERROR_CODE, START_TIMESTAMP, END_TIMESTAMP
FROM P360_DQ.AUDIT.STEP_RUN_LOG
WHERE RUN_ID = 'f04b01b7-d1ff-485f-9e00-4bbd6352d88b'
ORDER BY START_TIMESTAMP;

In [ ]:
-- Full pipeline test: reset everything and run FULL
UPDATE P360_DQ.BRONZE.STG_NPI_REGISTRY SET DQ_STATUS = NULL;
UPDATE P360_DQ.BRONZE.STG_SPECIALTY_TYPE SET DQ_STATUS = NULL;
TRUNCATE TABLE P360_DQ.SILVER.DIM_PROVIDER;
TRUNCATE TABLE P360_DQ.AUDIT.DQ_RESULT;
TRUNCATE TABLE P360_DQ.AUDIT.DQ_LOG;
CALL P360_DQ.ORCHESTRATION.SP_RUN_PACKAGE('FULL');

In [ ]:
%%sql -r debug_log2
SELECT * FROM P360_DQ.AUDIT.DQ_LOG ORDER BY DQ_START_TS DESC LIMIT 5;

In [ ]:
%%sql -r debug_status2
SELECT DQ_STATUS, COUNT(*) AS cnt FROM P360_DQ.BRONZE.STG_NPI_REGISTRY GROUP BY DQ_STATUS;

In [ ]:
%%sql -r final_bronze_status
-- Final validation: DQ_STATUS in Bronze
SELECT 'STG_NPI_REGISTRY' AS TBL, DQ_STATUS, COUNT(*) AS CNT FROM P360_DQ.BRONZE.STG_NPI_REGISTRY GROUP BY DQ_STATUS
UNION ALL
SELECT 'STG_SPECIALTY_TYPE', DQ_STATUS, COUNT(*) FROM P360_DQ.BRONZE.STG_SPECIALTY_TYPE GROUP BY DQ_STATUS
ORDER BY TBL, DQ_STATUS;

In [ ]:
%%sql -r final_dim_count
-- Final validation: DIM_PROVIDER populated
SELECT COUNT(*) AS ROW_COUNT FROM P360_DQ.SILVER.DIM_PROVIDER;

In [ ]:
%%sql -r final_dq_failures
-- Final validation: DQ_RESULT summary
SELECT TABLE_NM, RULE_ID, COUNT(*) AS FAIL_COUNT FROM P360_DQ.AUDIT.DQ_RESULT WHERE RESULT = 'FAIL' GROUP BY TABLE_NM, RULE_ID ORDER BY TABLE_NM, RULE_ID;

In [ ]:
%%sql -r step_order
-- Check step execution order
SELECT STEP_ID, STEP_ORDER, STEP_NAME, STEP_PROCEDURE, IS_ACTIVE
FROM P360_DQ.CONFIG.PKG_STEP_REGISTRY
WHERE IS_ACTIVE = TRUE
ORDER BY STEP_ORDER;

In [ ]:
%%sql -r pass_records
-- Check the NPI records that should flow to Silver (PASS + PASS-SOFT)
SELECT NPI_NUMBER, PROVIDER_NAME, DQ_STATUS
FROM P360_DQ.BRONZE.STG_NPI_REGISTRY
WHERE DQ_STATUS IN ('PASS', 'PASS-SOFT')
LIMIT 15;

In [ ]:
%%sql -r step_log_latest
-- Check step log for most recent run
SELECT STEP_NAME, STEP_STATUS, ROWS_READ, ROWS_WRITTEN, ERROR_CODE
FROM P360_DQ.AUDIT.STEP_RUN_LOG
WHERE RUN_ID = (SELECT MAX(RUN_ID) FROM P360_DQ.AUDIT.PKG_RUN_LOG)
ORDER BY START_TIMESTAMP;

In [ ]:
%%sql -r manual_silver_load
-- Manually call Silver load to verify it picks up PASS records
TRUNCATE TABLE P360_DQ.SILVER.DIM_PROVIDER;
CALL P360_DQ.SILVER.SP_LOAD_DIM_PROVIDER('manual-test', 'FULL');